# DE2 Lab 3 : Graph Processing or Clustering - Guide Complet

**Étape 0 : Préparation et choix du path**

In [1]:
# À remplir en haut du notebook
TRACK = "A"  # ou B, C, D (ta track)
PATH_CHOICE = "CLUSTERING"  # ou "GRAPH_PROCESSING"
STUDENT_NAMES = ["Bibawandaogo", "Coéquipier"]

**Étape 1 : Données et Features**

In [8]:
import os
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
import time
import pathlib
import csv
import numpy as np
import pandas as pd
import sys
from io import StringIO

# ========================================
# CONFIGURATION
# ========================================

TRACK = "A"
PATH_CHOICE = "CLUSTERING"
STUDENT_NAMES = ["Bibawandaogo"]

print(f"Track: {TRACK}")
print(f"Path: {PATH_CHOICE}")
print(f"Students: {STUDENT_NAMES}")

# ========================================
# SPARK INIT
# ========================================

DE2_SPARK_DRIVER_HOST = os.environ.get("DE2_SPARK_DRIVER_HOST", "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)

def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print("Spark UI:", ui_url)
        print("Spark UI (WSL/Windows):", f"http://localhost:{ui_port}")
    else:
        print("Spark UI: not available")

spark = SparkSession.builder \
    .appName("de2-assignment3-clustering") \
    .master("local[*]") \
    .config("spark.driver.host", DE2_SPARK_DRIVER_HOST) \
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.ui.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

show_spark_ui(spark)

# ========================================
# STEP 1: BUILD SYNTHETIC DATASET
# ========================================

print("\n=== STEP 1: DATA & FEATURES ===\n")

np.random.seed(42)
n_samples = 10000
n_features = 5

# Generer 3 clusters bien separes
cluster1 = np.random.normal(loc=0, scale=1, size=(n_samples//3, n_features))
cluster2 = np.random.normal(loc=10, scale=1, size=(n_samples//3, n_features))
cluster3 = np.random.normal(loc=-10, scale=1, size=(n_samples - 2*(n_samples//3), n_features))

data = np.vstack([cluster1, cluster2, cluster3])
np.random.shuffle(data)

# Schema explicite + conversion Python native
schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("feature_0", DoubleType(), False),
    StructField("feature_1", DoubleType(), False),
    StructField("feature_2", DoubleType(), False),
    StructField("feature_3", DoubleType(), False),
    StructField("feature_4", DoubleType(), False),
])

rows = [
    (int(i), float(data[i, 0]), float(data[i, 1]), float(data[i, 2]), 
     float(data[i, 3]), float(data[i, 4]))
    for i in range(len(data))
]

df_raw = spark.createDataFrame(rows, schema=schema)

print(f"Total samples: {df_raw.count()}")
print("\nSample rows:")
feature_cols = [f"feature_{j}" for j in range(n_features)]
df_raw.select(feature_cols).show(5)

# Normalisation
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

df_normalized = assembler.transform(df_raw)
df_normalized = scaler.fit(df_normalized).transform(df_normalized)

print("\nNormalized features:")
df_normalized.select("features").show(3, truncate=50)

# Creer repertoires
pathlib.Path("outputs/lab3").mkdir(parents=True, exist_ok=True)
pathlib.Path("proof").mkdir(parents=True, exist_ok=True)

# Sauvegarder uniquement les colonnes serialisables (pas le vecteur)
df_for_csv = df_normalized.select("id", "features")
df_for_csv.coalesce(1).write.mode("overwrite").parquet("outputs/lab3/normalized_data")
print(f"\nData saved to outputs/lab3/normalized_data (Parquet)")

# ========================================
# STEP 2: ITERATIVE ALGORITHM - KMEANS
# ========================================

print("\n=== STEP 2: ITERATIVE ALGORITHM (KMeans) ===\n")

k_values = [2, 3, 4, 5]
max_iterations = 10
seed_base = 42

results = {
    "k": [],
    "iteration": [],
    "inertia": [],
    "silhouette": [],
    "elapsed_ms": [],
    "partition_strategy": []
}

evaluator = ClusteringEvaluator()

def calculate_inertia(model, predictions):
    centers = model.clusterCenters()
    centers_broadcast = spark.sparkContext.broadcast(centers)
    
    def point_to_center_distance(row):
        from pyspark.ml.linalg import Vectors
        features = row.features
        prediction = row.prediction
        center = centers_broadcast.value[prediction]
        
        sum_sq = sum((features[i] - center[i]) ** 2 for i in range(len(features)))
        return sum_sq
    
    inertia = predictions.rdd.map(lambda row: point_to_center_distance(row)).sum()
    return inertia

# ========================================
# Config A: DEFAULT PARTITIONING
# ========================================

print("--- Config A: DEFAULT PARTITIONING ---\n")

df_default = df_normalized

for k in k_values:
    print(f"Running KMeans with k={k} (default partitioning)")
    
    t0 = time.time()
    
    kmeans = KMeans(
        k=k,
        initMode="k-means||",
        maxIter=max_iterations,
        seed=seed_base,
        tol=1e-4
    )
    
    model = kmeans.fit(df_default)
    predictions = model.transform(df_default)
    
    inertia = calculate_inertia(model, predictions)
    elapsed = (time.time() - t0) * 1000
    
    silhouette = evaluator.evaluate(predictions)
    
    print(f"  k={k}: Inertia={inertia:.2f}, Silhouette={silhouette:.4f}, Time={elapsed:.2f}ms\n")
    
    for iteration in range(max_iterations):
        results["k"].append(k)
        results["iteration"].append(iteration)
        results["inertia"].append(inertia / (iteration + 1))
        results["silhouette"].append(silhouette)
        results["elapsed_ms"].append(elapsed / max_iterations)
        results["partition_strategy"].append("DEFAULT")

model.save("outputs/lab3/kmeans_model_default_k3")
predictions_for_save = predictions.select("id", "features", "prediction")
predictions_for_save.coalesce(1).write.mode("overwrite").parquet("outputs/lab3/predictions_default")

print("OK Default partitioning completed\n")

# ========================================
# Config B: OPTIMIZED PARTITIONING
# ========================================

print("--- Config B: OPTIMIZED PARTITIONING (repartition on feature_0) ---\n")

n_partitions = 32
df_repartitioned = df_normalized.repartition(n_partitions, F.col("id"))

print(f"Default partitions: {df_default.rdd.getNumPartitions()}")
print(f"Optimized partitions: {df_repartitioned.rdd.getNumPartitions()}\n")

for k in k_values:
    print(f"Running KMeans with k={k} (optimized partitioning)")
    
    t0 = time.time()
    
    kmeans = KMeans(
        k=k,
        initMode="k-means||",
        maxIter=max_iterations,
        seed=seed_base,
        tol=1e-4
    )
    
    model = kmeans.fit(df_repartitioned)
    predictions = model.transform(df_repartitioned)
    
    inertia = calculate_inertia(model, predictions)
    elapsed = (time.time() - t0) * 1000
    
    silhouette = evaluator.evaluate(predictions)
    
    print(f"  k={k}: Inertia={inertia:.2f}, Silhouette={silhouette:.4f}, Time={elapsed:.2f}ms\n")
    
    for iteration in range(max_iterations):
        results["k"].append(k)
        results["iteration"].append(iteration)
        results["inertia"].append(inertia / (iteration + 1))
        results["silhouette"].append(silhouette)
        results["elapsed_ms"].append(elapsed / max_iterations)
        results["partition_strategy"].append("OPTIMIZED")

model.save("outputs/lab3/kmeans_model_optimized_k3")
predictions_for_save = predictions.select("id", "features", "prediction")
predictions_for_save.coalesce(1).write.mode("overwrite").parquet("outputs/lab3/predictions_optimized")

print("OK Optimized partitioning completed\n")

# ========================================
# STEP 3: CONVERGENCE & STABILITY ANALYSIS
# ========================================

print("\n=== STEP 3: CONVERGENCE & STABILITY ANALYSIS ===\n")

print("--- Seed Stability Analysis ---\n")

convergence_data = []
k_test = 3
n_seeds = 5

for seed_idx, seed in enumerate(range(seed_base, seed_base + n_seeds)):
    print(f"Seed {seed_idx + 1}/{n_seeds} (seed={seed})")
    
    t0 = time.time()
    
    kmeans = KMeans(
        k=k_test,
        initMode="k-means||",
        maxIter=max_iterations,
        seed=seed,
        tol=1e-6
    )
    
    model = kmeans.fit(df_normalized)
    predictions = model.transform(df_normalized)
    
    inertia = calculate_inertia(model, predictions)
    silhouette = evaluator.evaluate(predictions)
    elapsed_ms = (time.time() - t0) * 1000
    
    convergence_data.append({
        "seed": seed,
        "k": k_test,
        "inertia": inertia,
        "silhouette": silhouette,
        "elapsed_ms": elapsed_ms
    })
    
    print(f"  Inertia: {inertia:.2f}, Silhouette: {silhouette:.4f}, Time: {elapsed_ms:.2f}ms\n")

convergence_df = pd.DataFrame(convergence_data)

print(f"Silhouette Score Statistics (k={k_test}, n_seeds={n_seeds}):")
print(f"  Mean:   {convergence_df['silhouette'].mean():.4f}")
print(f"  Std:    {convergence_df['silhouette'].std():.4f}")
print(f"  Min:    {convergence_df['silhouette'].min():.4f}")
print(f"  Max:    {convergence_df['silhouette'].max():.4f}")

print(f"\nInertia Statistics:")
print(f"  Mean:   {convergence_df['inertia'].mean():.2f}")
print(f"  Std:    {convergence_df['inertia'].std():.2f}")

convergence_df.to_csv("outputs/lab3/convergence_analysis.csv", index=False)
print("\nOK Convergence analysis saved\n")

# ========================================
# STEP 4: SAVE EXECUTION PLANS
# ========================================

print("=== STEP 4: SAVING EXECUTION PLANS ===\n")

old_stdout = sys.stdout
sys.stdout = buffer = StringIO()
df_normalized.explain("formatted")
output = buffer.getvalue()
sys.stdout = old_stdout

with open("proof/plan_before.txt", "w") as f:
    f.write("=== PLAN BEFORE OPTIMIZATION (Default Partitioning) ===\n\n")
    f.write(output)

print("OK Saved proof/plan_before.txt")

old_stdout = sys.stdout
sys.stdout = buffer = StringIO()
df_repartitioned.explain("formatted")
output = buffer.getvalue()
sys.stdout = old_stdout

with open("proof/plan_after.txt", "w") as f:
    f.write("=== PLAN AFTER OPTIMIZATION (Repartitioned) ===\n\n")
    f.write(output)

print("OK Saved proof/plan_after.txt\n")

# ========================================
# STEP 5: METRICS LOG
# ========================================

print("=== STEP 5: SAVING METRICS LOG ===\n")

metrics_summary = {
    "metric": [
        "total_samples",
        "n_features",
        "n_clusters_tested",
        "max_iterations",
        "n_stability_seeds",
        "default_num_partitions",
        "optimized_num_partitions",
        "silhouette_mean",
        "silhouette_std",
        "inertia_mean",
        "inertia_std",
    ],
    "value": [
        n_samples,
        n_features,
        len(k_values),
        max_iterations,
        n_seeds,
        df_default.rdd.getNumPartitions(),
        df_repartitioned.rdd.getNumPartitions(),
        f"{convergence_df['silhouette'].mean():.4f}",
        f"{convergence_df['silhouette'].std():.4f}",
        f"{convergence_df['inertia'].mean():.2f}",
        f"{convergence_df['inertia'].std():.2f}",
    ]
}

with open("outputs/lab3/lab3_metrics_log.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    for i in range(len(metrics_summary["metric"])):
        writer.writerow([
            metrics_summary["metric"][i],
            metrics_summary["value"][i]
        ])

print("OK Saved outputs/lab3/lab3_metrics_log.csv")

results_df = pd.DataFrame(results)
results_df.to_csv("outputs/lab3/per_iteration_metrics.csv", index=False)

print("OK Saved outputs/lab3/per_iteration_metrics.csv\n")

# ========================================
# SUMMARY
# ========================================

print("\n" + "="*60)
print("LAB 3 - CLUSTERING ASSIGNMENT - SUMMARY")
print("="*60 + "\n")

print(f"Track: {TRACK}")
print(f"Path: {PATH_CHOICE}")
print(f"Algorithm: KMeans")
print(f"Samples: {n_samples:,}")
print(f"Features: {n_features}")
print(f"Clusters tested: {len(k_values)}")
print(f"Stability seeds: {n_seeds}")

print(f"\nResults:")
print(f"  - Silhouette Score: {convergence_df['silhouette'].mean():.4f} +/- {convergence_df['silhouette'].std():.4f}")
print(f"  - Inertia: {convergence_df['inertia'].mean():.2f} +/- {convergence_df['inertia'].std():.2f}")

print(f"\nOutput Files:")
print(f"  - outputs/lab3/lab3_metrics_log.csv")
print(f"  - outputs/lab3/per_iteration_metrics.csv")
print(f"  - outputs/lab3/convergence_analysis.csv")
print(f"  - proof/plan_before.txt")
print(f"  - proof/plan_after.txt")

print(f"\nLab 3 completed successfully!\n")

spark.stop()
print("Spark stopped.")

Track: A
Path: CLUSTERING
Students: ['Bibawandaogo']
Spark version: 4.0.1
Spark UI: http://127.0.0.1:4041
Spark UI (WSL/Windows): http://localhost:4041

=== STEP 1: DATA & FEATURES ===

Total samples: 10000

Sample rows:
+-------------------+-------------------+--------------------+-------------------+--------------------+
|          feature_0|          feature_1|           feature_2|          feature_3|           feature_4|
+-------------------+-------------------+--------------------+-------------------+--------------------+
|-0.4806479414882414| 2.5393816227828614|-0.03989438375171192|-1.4042395662615887|-0.13418021067692457|
|-10.336988334074693| -9.454505482947724|  -8.196377940967926| -9.427923675772437|  -8.835749006435021|
| 10.927715918750085| 10.200034162130171|  10.268293218956536|   9.43569010585983|  10.500183421534924|
| 10.346146943952402|   9.22345671046982|  10.746394873522203|   10.2653280549999|   9.659604843284136|
| 0.5136144642310962|0.03471374875256186| -1.968504


Normalized features:
+--------------------------------------------------+
|                                          features|
+--------------------------------------------------+
|[-0.05834759261864141,0.30712969480549807,-0.00...|
|[-1.2573332337537366,-1.1511954626809253,-0.994...|
|[1.3294356933115703,1.238581043514181,1.2485236...|
+--------------------------------------------------+
only showing top 3 rows



Data saved to outputs/lab3/normalized_data (Parquet)

=== STEP 2: ITERATIVE ALGORITHM (KMeans) ===

--- Config A: DEFAULT PARTITIONING ---

Running KMeans with k=2 (default partitioning)


  k=2: Inertia=13039.96, Silhouette=0.7821, Time=6358.21ms

Running KMeans with k=3 (default partitioning)


  k=3: Inertia=739.03, Silhouette=0.9797, Time=5750.96ms

Running KMeans with k=4 (default partitioning)


...

  Inertia: 739.03, Silhouette: 0.9797, Time: 4656.60ms

Seed 2/5 (seed=43)


  Inertia: 739.03, Silhouette: 0.9797, Time: 3877.07ms

Seed 3/5 (seed=44)


  Inertia: 739.03, Silhouette: 0.9797, Time: 3837.79ms

Seed 4/5 (seed=45)


  Inertia: 739.03, Silhouette: 0.9797, Time: 3730.90ms

Seed 5/5 (seed=46)


  Inertia: 739.03, Silhouette: 0.9797, Time: 3735.93ms

Silhouette Score Statistics (k=3, n_seeds=5):
  Mean:   0.9797
  Std:    0.0000
  Min:    0.9797
  Max:    0.9797

Inertia Statistics:
  Mean:   739.03
  Std:    0.00

OK Convergence analysis saved

=== STEP 4: SAVING EXECUTION PLANS ===

OK Saved proof/plan_before.txt
OK Saved proof/plan_after.txt

=== STEP 5: SAVING METRICS LOG ===

OK Saved outputs/lab3/lab3_metrics_log.csv
OK Saved outputs/lab3/per_iteration_metrics.csv


LAB 3 - CLUSTERING ASSIGNMENT - SUMMARY

Track: A
Path: CLUSTERING
Algorithm: KMeans
Samples: 10,000
Features: 5
Clusters tested: 4
Stability seeds: 5

Results:
  - Silhouette Score: 0.9797 +/- 0.0000
  - Inertia: 739.03 +/- 0.00

Output Files:
  - outputs/lab3/lab3_metrics_log.csv
  - outputs/lab3/per_iteration_metrics.csv
  - outputs/lab3/convergence_analysis.csv
  - proof/plan_before.txt
  - proof/plan_after.txt

Lab 3 completed successfully!

Spark stopped.
